**Feature Selection Method:** FLAML   
**Dataset:** QUT-DV25 (Opensnoop)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif
from flaml import AutoML

In [ ]:
# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
DATASET_NAME = "Opensnoop"
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:
gc.collect()
tf.keras.backend.clear_session()


In [ ]:
data.head()

,Package_Name,Total_Paths,Total_Error,Total_ File_Descriptor,Python_Related_Keywords,Install_Package_Keywords,Root_DIR_Installation,Temporary_DIR_Installation,Home_DIR_Installation,User_Access,Sys_Access,Etc_DIR_Installation,Other_DIR_Installation,Level
0,10Cent10-999.0.4.tar.gz,8045,8045,29,1083,2305,0,71,792,411,1225,151,5395,1
1,10Cent11-999.0.4.tar.gz,4790,4790,12,1616,843,2,87,1118,721,1200,104,1558,1
2,11Cent-999.0.0.tar.gz,26159,26159,27,3299,4536,0,200,2485,1072,1236,450,20716,1
3,11Cent-999.0.1.tar.gz,11194,11194,28,1521,2558,2,97,1035,891,1211,265,7693,1
4,11Cent-999.0.2.tar.gz,13561,13561,35,2656,5265,501,17,1497,1719,1219,291,8317,1


In [ ]:

all_pattern_columns = ['Total_Paths', 'Total_Error', 'Total_ File_Descriptor',
       'Python_Related_Keywords', 'Install_Package_Keywords',
       'Root_DIR_Installation', 'Temporary_DIR_Installation',
       'Home_DIR_Installation', 'User_Access', 'Sys_Access',
       'Etc_DIR_Installation', 'Other_DIR_Installation', 'Level']


data[all_pattern_columns] = data[all_pattern_columns].fillna(0)


# Assume 'data' is your pandas DataFrame
X = data.drop(columns=['Level'])
y = data['Level']

if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize FLAML
automl = AutoML()
automl_settings = {
    "time_budget": 300,
    "metric": "accuracy",
    "task": "classification",
}

automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

# Feature importance & selection
selected_features = X.columns.tolist()
dropped_features = []

try:
    importances = automl.model.feature_importances_

    # Sometimes LGBM may drop unused features -> align with all columns
    if len(importances) < X.shape[1]:
        full_importances = np.zeros(X.shape[1])
        # Use feature names from model if available
        try:
            model_features = automl.model.booster_.feature_name()
            for i, col in enumerate(X.columns):
                if col in model_features:
                    idx = model_features.index(col)
                    full_importances[i] = importances[idx]
        except:
            full_importances[:len(importances)] = importances
        importances = full_importances

    varimp = pd.DataFrame({'variable': X.columns, 'importance': importances})
    varimp['relative_importance'] = varimp['importance'] / varimp['importance'].sum()

    threshold = 0.03
    selected_features = list(varimp[varimp['relative_importance'] > threshold]['variable'])
    dropped_features = list(varimp[varimp['relative_importance'] <= threshold]['variable'])

    print("Feature importance table:\n", varimp)

except AttributeError:
    print("Feature importance not available for the chosen model.")

print("\nSelected features:", selected_features)
print("Number of features kept:", len(selected_features))
print("\nDropped features:", dropped_features)
print("Number of features dropped:", len(dropped_features))


[flaml.automl.logger: 08-29 21:14:54] {1680} INFO - task = classification
[flaml.automl.logger: 08-29 21:14:54] {1691} INFO - Evaluation method: cv
[flaml.automl.logger: 08-29 21:14:54] {1789} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 08-29 21:14:54] {1901} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'lrl1']
[flaml.automl.logger: 08-29 21:14:54] {2219} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 08-29 21:14:54] {2346} INFO - Estimated sufficient time budget=660s. Estimated necessary time budget=15s.
[flaml.automl.logger: 08-29 21:14:54] {2398} INFO -  at 0.1s,	estimator lgbm's best error=0.2132,	best estimator lgbm's best error=0.2132
[flaml.automl.logger: 08-29 21:14:54] {2219} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 08-29 21:14:54] {2398} INFO -  at 0.2s,	estimator lgbm's best error=0.2132,	best estimator lgbm's best error=0.2132
[flaml.automl.logger: 08-29 21:14:5

[flaml.automl.logger: 08-29 21:15:22] {2398} INFO -  at 28.6s,	estimator lgbm's best error=0.0517,	best estimator lgbm's best error=0.0517
[flaml.automl.logger: 08-29 21:15:22] {2219} INFO - iteration 34, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 21:15:23] {2398} INFO -  at 29.2s,	estimator xgb_limitdepth's best error=0.0606,	best estimator lgbm's best error=0.0517
[flaml.automl.logger: 08-29 21:15:23] {2219} INFO - iteration 35, current learner lgbm
[flaml.automl.logger: 08-29 21:15:26] {2398} INFO -  at 32.6s,	estimator lgbm's best error=0.0506,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:15:26] {2219} INFO - iteration 36, current learner rf
[flaml.automl.logger: 08-29 21:15:28] {2398} INFO -  at 34.0s,	estimator rf's best error=0.1586,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:15:28] {2219} INFO - iteration 37, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 21:15:28] {2398} INFO -  at 34.6s,	estimat

C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


[flaml.automl.logger: 08-29 21:15:57] {2398} INFO -  at 63.7s,	estimator lrl1's best error=0.3450,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:15:57] {2219} INFO - iteration 60, current learner lrl1


C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


...

[flaml.automl.logger: 08-29 21:16:05] {2398} INFO -  at 71.7s,	estimator lrl1's best error=0.3449,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:16:05] {2219} INFO - iteration 66, current learner lgbm


C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


[flaml.automl.logger: 08-29 21:16:08] {2398} INFO -  at 74.7s,	estimator lgbm's best error=0.0506,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:16:08] {2219} INFO - iteration 67, current learner xgboost
[flaml.automl.logger: 08-29 21:16:09] {2398} INFO -  at 75.1s,	estimator xgboost's best error=0.0640,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:16:09] {2219} INFO - iteration 68, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 21:16:09] {2398} INFO -  at 75.4s,	estimator xgb_limitdepth's best error=0.0606,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:16:09] {2219} INFO - iteration 69, current learner xgboost
[flaml.automl.logger: 08-29 21:16:13] {2398} INFO -  at 79.4s,	estimator xgboost's best error=0.0562,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:16:13] {2219} INFO - iteration 70, current learner lgbm
[flaml.automl.logger: 08-29 21:16:22] {2398} INFO -  at 88.5s,	e

[flaml.automl.logger: 08-29 21:17:51] {2398} INFO -  at 177.8s,	estimator rf's best error=0.0851,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:17:51] {2219} INFO - iteration 102, current learner xgboost
[flaml.automl.logger: 08-29 21:18:09] {2398} INFO -  at 195.9s,	estimator xgboost's best error=0.0545,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:18:09] {2219} INFO - iteration 103, current learner rf
[flaml.automl.logger: 08-29 21:18:12] {2398} INFO -  at 198.2s,	estimator rf's best error=0.0743,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:18:12] {2219} INFO - iteration 104, current learner lgbm
[flaml.automl.logger: 08-29 21:18:17] {2398} INFO -  at 203.5s,	estimator lgbm's best error=0.0506,	best estimator lgbm's best error=0.0506
[flaml.automl.logger: 08-29 21:18:17] {2219} INFO - iteration 105, current learner rf
[flaml.automl.logger: 08-29 21:18:19] {2398} INFO -  at 205.9s,	estimator rf's best error=

[flaml.automl.logger: 08-29 21:19:59] {1931} INFO - fit succeeded
[flaml.automl.logger: 08-29 21:19:59] {1932} INFO - Time taken to find the best model: 286.68995666503906
Feature importance table:
                       variable  importance  relative_importance
0                 Package_Name      5634.0             0.085955
1                  Total_Paths      2439.0             0.037211
2                  Total_Error      4751.0             0.072483
3       Total_ File_Descriptor      5722.0             0.087297
4      Python_Related_Keywords      6083.0             0.092805
5     Install_Package_Keywords      2899.0             0.044228
6        Root_DIR_Installation      6071.0             0.092622
7   Temporary_DIR_Installation      6638.0             0.101272
8        Home_DIR_Installation      6674.0             0.101822
9                  User_Access      7489.0             0.114256
10                  Sys_Access      5984.0             0.091295
11        Etc_DIR_Installation   

In [ ]:
selected_features = ['Total_Paths', 'Total_Error', 'Total_ File_Descriptor', 'Python_Related_Keywords', 'Install_Package_Keywords', 'Root_DIR_Installation', 'Temporary_DIR_Installation', 'Home_DIR_Installation', 'User_Access', 'Sys_Access', 'Etc_DIR_Installation']